## 依赖导入

本例演示偏最小二乘（PLS）回归：读取原始数据矩阵、用 `zscore` 标准化、以交叉验证均方误差选择成分个数，再输出成分得分、载荷以及标准化/原始尺度的回归系数。依赖 `scipy.stats.zscore` 与 `sklearn` 的 `PLSRegression`、`cross_val_predict`、`mean_squared_error`。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_predict

## 全局绘图配置

设置全局 `plt.rc`：中文字体 SimHei、字号 15、关闭坐标轴负号的 Unicode 显示，保证后续回归系数柱状图与“预测—观测”散点图的中文标签正常显示。

In [ ]:
plt.rc("font", family="SimHei")
plt.rc("axes", unicode_minus=False)
plt.rc("font", size=15)

## 11.1  偏最小二乘回归的成分数选择与回归系数

读取 `data/13_1.txt`，先求均值 `mu`、标准差 `s`，再用 `zscore` 标准化得 `d`，前三列为自变量 $X$、后三列为因变量 $Y$。对成分数 $1\sim n$ 逐一做交叉验证计算均方误差 `rmse`，取其最小值给出建议成分数（源程序 ex13_1.py）。随后拟合两成分 PLS 模型 `md`，输出 $X/Y$ 的成分得分 `xd/yd`、用 `pinv` 求得的自变量与因变量成分系数 `zx/zy`、主成分载荷 `xzh/yzh` 及标准化回归系数 `beta2`（保留 4 位）；再按均值/标准差把标准化系数换算成原始尺度的回归系数矩阵 `beta3`（首行为截距项），构造设计矩阵 `aa` 求出预测值 `yh`。最后绘制一幅按三个成分分组的回归系数柱状图（横轴标注“单杠/弯曲/跳高”），并用三幅子图分别给出各因变量的“预测—观测”散点及 $y=x$ 对角参照线。

In [ ]:
d0 = np.loadtxt("data/13_1.txt")
N = d0.shape[0]
mu = d0.mean(axis=0)
s = d0.std(axis=0, ddof=1)
d = zscore(d0, ddof=1)
a = d[:, :3]
b = d[:, 3:]
n = a.shape[1]
m = b.shape[1]
rmse = []
for i in range(1, n + 1):
    pls = PLSRegression(i)
    y_cv = cross_val_predict(pls, a, b)
    rmse.append(mean_squared_error(b, y_cv))
nmin = np.argmin(rmse)
print("均方误差：\n", rmse)
print("建议的成分个数: ", nmin + 1)

md = PLSRegression(2).fit(a, b)
xd = md.x_scores_
yd = md.y_scores_
zx = np.linalg.pinv(a) @ xd
print("自变量的成分系数 (列）：\n", zx)
zy = np.linalg.pinv(b) @ yd
print("因变量的成分系数 (列)：\n", zy)
xzh = md.x_loadings_
yzh = md.y_loadings_
print("x 主成分回归 (行)：\n", xzh)
print("\n------")
print("y 主成分回归 (行)：\n", yzh)
print("\n------")
beta2 = md.coef_
print("（标准化）y 关于 x 回归系数 (列)：\n", np.round(beta2, 4))
beta3 = np.zeros((n + 1, m))
beta3[0, :] = mu[n:] - mu[:n] / s[:n] @ beta2 * s[n:]
for i in range(m):
    beta3[1:, i] = s[n + i] / s[:n] * beta2[:, i]
print("(原始数据)y 关于 x 回归系数 (列):\n", beta3)
aa = np.hstack([np.ones((N, 1)), d0[:, :n]])
yh = aa @ beta3

x0 = np.arange(1, 4)
fig, ax = plt.subplots()
ax.bar(x0, beta2[0, :], 0.1)
ax.bar(x0 + 0.1, beta2[1, :], 0.1)
ax.bar(x0 + 0.2, beta2[2, :], 0.1)
ax.set_xticks(x0 + 0.1, labels=["单杠", "弯曲", "跳高"])
b0 = d0[:, 3:]
mb = np.max(b0, axis=0)
fig2, (ax2, ax3, ax4) = plt.subplots(1, 3)
ax2.plot(yh[:, 0], d0[:, 3], "o", [0, mb[0]], [0, mb[0]])
ax2.set(xlabel="预测数据", ylabel="观测数据")
ax3.plot(yh[:, 1], d0[:, 4], "*", [0, mb[1]], [0, mb[1]])
ax3.set(xlabel="预测数据", ylabel="观测数据")
ax4.plot(yh[:, 2], d0[:, 5], "p", [0, mb[2]], [0, mb[2]])
ax4.set(xlabel="预测数据", ylabel="观测数据")
plt.show()

## 11.2  偏最小二乘回归系数（标准化与原始尺度）

读取 `data/13_2.txt`，求均值、标准差并用 `zscore` 标准化；前七列为自变量、其后为因变量。用交叉验证均方误差在 $1\sim n$ 个成分中选择并给出建议成分数（源程序 ex13_2.py）。拟合两成分 PLS 后输出标准化数据的回归系数 `coef_`，并按均值/标准差换算为原始尺度的回归系数 `b0`（首行为截距项）。

In [ ]:
d0 = np.loadtxt("data/13_2.txt")
mu = d0.mean(axis=0)
s = d0.std(axis=0, ddof=1)
d = zscore(d0, ddof=1)
a = d[:, :7]
b = d[:, 7:]
n = a.shape[1]
m = b.shape[1]
mse = []
for i in range(1, n + 1):
    pls = PLSRegression(i)
    y_cv = cross_val_predict(pls, a, b)
    mse.append(mean_squared_error(b, y_cv))
nmin = np.argmin(mse)
print("均方误差：\n", mse)
print("建议的成分个数: ", nmin + 1)
md = PLSRegression(2).fit(a, b)
b = md.coef_
print("标准化数据的回归系数 (列)：\n", b)
b0 = np.zeros((n + 1, m))
b0[0, :] = mu[n:] - mu[:n] / s[:n] @ b * s[n:]
for i in range(m):
    b0[1:, i] = s[n + i] / s[:n] * b[:, i]
print("(原始数据)y 关于 x 回归系数 (列):\n", b0)